# Mastering Transfer Learning
### A hands-on notebook aligned with the Transfer Learning roadmap

This notebook is a **teaching artifact**: each section corresponds directly to a concept
in the roadmap, in the same conceptual order:

1. **Pretraining (Knowledge Acquisition)** — using a pre-trained model, defining the
   source domain $D_s$ and source task $T_s$.
2. **Target domain & task** — defining $D_t$ and $T_t$, and where they diverge from
   $D_s, T_s$ (the reason transfer learning is needed at all).
3. **Feature Reuse** — network modification: why low-level features transfer, and how
   we swap the output head for the target task's output space.
4. **Freezing & Fine-Tuning strategies** — feature extraction (frozen backbone),
   full fine-tuning, partial unfreezing (controlled exception), and a PEFT/adapter demo.
5. **Task & Domain Relatedness** — an empirical comparison that demonstrates *why*
   relatedness matters, and a discussion of **negative transfer**.

> **Dataset note:** We use `torchvision`'s pretrained `ResNet18` (source: ImageNet,
> 1000-way classification) as $D_s, T_s$, and a small subset of `CIFAR-10` as the
> target domain/task $D_t, T_t$. The subsetting keeps every cell runnable in a few
> minutes on CPU, while still producing meaningful comparative curves. Swap in your
> own dataset by editing the `Target dataset` section only — nothing downstream needs
> to change.

## 0. Definitions & Assumptions (reference)

**Formal definition.** Given a source domain $D_s = \{\mathcal{X}_s, P(X_s)\}$ and
source task $T_s = \{\mathcal{Y}_s, f_s(\cdot)\}$, and a target domain
$D_t = \{\mathcal{X}_t, P(X_t)\}$ and target task $T_t = \{\mathcal{Y}_t, f_t(\cdot)\}$,
transfer learning improves the target predictive function $f_t(\cdot)$ using knowledge
from $D_s$ and $T_s$, where $D_s \neq D_t$ or $T_s \neq T_t$.

**Three assumptions we will keep testing against the code below:**

| Assumption | What it means here |
|---|---|
| Feature Transferability | Low-level ResNet18 filters (edges/textures) should be reusable on CIFAR-10 images |
| Data Asymmetry | ImageNet (source) is huge; our CIFAR-10 subset (target) is deliberately small |
| Task/Domain Relatedness | Both are natural-image classification -> related. We will also simulate an *unrelated* setup to show negative transfer risk |

## 1. Setup

In [1]:
# Core imports: torch/torchvision for models+data, matplotlib for the final comparison plots.
import copy
import time
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 2. Pretraining (Knowledge Acquisition) — the Source Domain $D_s$ and Task $T_s$

Roadmap alternative chosen here: **"Using a pre-trained model"** — we leverage an
open-source model (`ResNet18`) trained on a massive, generalized dataset (ImageNet)
instead of building a source model from scratch.

- $D_s$: natural RGB images, $\mathcal{X}_s \subset \mathbb{R}^{224\times224\times3}$,
  $P(X_s)$ = ImageNet's image distribution.
- $T_s$: $\mathcal{Y}_s$ = 1000 ImageNet classes, $f_s(\cdot)$ = the trained ResNet18
  mapping images to those 1000 classes.

In [2]:
# Load a ResNet18 pretrained on ImageNet -> this IS f_s(.), the source predictive function.
source_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
source_model.eval()

# Quick architectural inventory: convolutional "backbone" (feature extractor) vs
# the final "head" (fully-connected classifier specific to the source task's 1000 classes).
backbone_params = sum(p.numel() for name, p in source_model.named_parameters() if not name.startswith("fc"))
head_params = sum(p.numel() for name, p in source_model.named_parameters() if name.startswith("fc"))

print("Source model: torchvision ResNet18 (ImageNet1K weights)")
print(f"  Backbone (feature extractor) parameters: {backbone_params:,}")
print(f"  Head (fc, 1000-way classifier) parameters: {head_params:,}")
print(f"  Head shape: {source_model.fc}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 135MB/s]


Source model: torchvision ResNet18 (ImageNet1K weights)
  Backbone (feature extractor) parameters: 11,176,512
  Head (fc, 1000-way classifier) parameters: 513,000
  Head shape: Linear(in_features=512, out_features=1000, bias=True)


## 3. Target Domain $D_t$ and Task $T_t$

- $D_t$: CIFAR-10 images, natively $32\times32\times3$ — a **different distribution**
  $P(X_t) \neq P(X_s)$ (lower resolution, different object framing/scale) even though
  both are "natural images". This is exactly the $D_s \neq D_t$ condition in the
  definition.
- $T_t$: $\mathcal{Y}_t$ = 10 CIFAR-10 classes, so $T_s \neq T_t$ as well (different
  label space, different $f_t(\cdot)$ to learn).

We deliberately keep the target set **small** (a few hundred images per class) to
reflect the roadmap's *Data Asymmetry* motivation: abundant source data, scarce target
data. We resize/normalize to match the statistics ResNet18 was trained on, since the
backbone's early filters expect that input distribution.

In [ ]:
# Preprocessing must match the source model's training distribution (ImageNet mean/std,
# 224x224 input) so the transferred low-level filters remain meaningful on the target data.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

target_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_train = datasets.CIFAR10(root="./data", train=True, download=True, transform=target_transform)
full_test = datasets.CIFAR10(root="./data", train=False, download=True, transform=target_transform)

# Simulate a SMALL target dataset (data asymmetry): a few hundred images/class for train,
# a modest fixed-size set for validation. This keeps every experiment below fast to run
# while still being large enough to show clear differences between strategies.
def stratified_subset(dataset, per_class, num_classes=10, seed=SEED):
    rng = np.random.RandomState(seed)
    targets = np.array(dataset.targets)
    idx = []
    for c in range(num_classes):
        class_idx = np.where(targets == c)[0]
        chosen = rng.choice(class_idx, size=per_class, replace=False)
        idx.extend(chosen.tolist())
    rng.shuffle(idx)
    return Subset(dataset, idx)

train_subset = stratified_subset(full_train, per_class=60)   # ~600 target training images
val_subset = stratified_subset(full_test, per_class=40)      # ~400 target validation images

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False, num_workers=0)

print(f"Target train set: {len(train_subset)} images | Target val set: {len(val_subset)} images")
print(f"Target classes (Y_t): {full_train.classes}")

 37%|███▋      | 62.3M/170M [08:24<14:13, 127kB/s]

## 4. Feature Reuse — Network Modification

Two roadmap ideas land here:

- **Feature Transferability assumption**: low-level conv filters (edges, textures,
  color blobs) generalize across natural-image domains, so we keep the ResNet18
  backbone as-is.
- **Network Modification**: the output layer trained for $\mathcal{Y}_s$ (1000
  classes) is removed and replaced with a new layer matching $\mathcal{Y}_t$
  (10 classes).

In [ ]:
def build_target_model(pretrained=True):
    '''Return a ResNet18 with its head replaced for the 10-class target task.
    pretrained=False gives a randomly-initialized backbone, used later as a
    no-transfer control for the relatedness experiment.'''
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    in_features = model.fc.in_features
    # Network Modification: swap the 1000-way ImageNet head for a fresh 10-way head.
    model.fc = nn.Linear(in_features, 10)
    return model.to(device)

demo_model = build_target_model(pretrained=True)
print("New head for the target task (Y_t, 10 classes):")
print(demo_model.fc)
print("\nBackbone (feature extractor) reused unchanged from the source model.")
del demo_model

## 5. Adaptation — Freezing & Fine-Tuning Strategies

A single reusable training/eval loop lets us compare strategies fairly. We will run:

- **A. Freeze backbone, train head only** — recommended for very small target datasets.
- **B. Full fine-tuning** — recommended when more target data is available.
- **C. Partial unfreezing (last block only)** — the roadmap's controlled exception,
  useful under class imbalance or domain shift, kept deliberate and monitored.
- **D. PEFT-style adapters** — small injected modules trained instead of the whole
  network, preserving original weights while adapting cheaply.

In [ ]:
def train_and_evaluate(model, train_loader, val_loader, epochs=5, lr=1e-3, label=""):
    '''Trains only the parameters with requires_grad=True (so the caller controls
    what is frozen), and returns per-epoch train loss + validation accuracy history.'''
    criterion = nn.CrossEntropyLoss()
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(trainable_params, lr=lr)

    n_trainable = sum(p.numel() for p in trainable_params)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"[{label}] training {n_trainable:,}/{n_total:,} parameters "
          f"({100*n_trainable/n_total:.1f}%)")

    history = {"train_loss": [], "val_acc": []}
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["val_acc"].append(val_acc)
        print(f"  [{label}] epoch {epoch+1}/{epochs} - train_loss={train_loss:.4f} - val_acc={val_acc:.3f}")

    return history

In [ ]:
# --- Strategy A: Freeze backbone, train head only -----------------------------------
# Recommended when the target dataset is very small (our ~600-image case).
model_frozen = build_target_model(pretrained=True)
for name, param in model_frozen.named_parameters():
    param.requires_grad = name.startswith("fc")   # only the new head is trainable

history_frozen = train_and_evaluate(
    model_frozen, train_loader, val_loader, epochs=5, lr=1e-3, label="A: frozen backbone"
)

In [ ]:
# --- Strategy B: Full fine-tuning ----------------------------------------------------
# Recommended when more target data/compute is available; here we use a lower LR since
# ALL weights (including the pretrained backbone) are updated, to avoid destroying
# the transferred features too quickly ("catastrophic forgetting").
model_full = build_target_model(pretrained=True)
for param in model_full.parameters():
    param.requires_grad = True

history_full = train_and_evaluate(
    model_full, train_loader, val_loader, epochs=5, lr=1e-4, label="B: full fine-tune"
)

In [ ]:
# --- Strategy C: Partial unfreezing (last block only) --------------------------------
# The roadmap's controlled exception: justified under class imbalance / domain shift,
# done deliberately and monitored for overfitting. We unfreeze only ResNet18's last
# residual block ("layer4") plus the new head, keeping earlier (more generic) layers frozen.
model_partial = build_target_model(pretrained=True)
for name, param in model_partial.named_parameters():
    param.requires_grad = name.startswith("fc") or name.startswith("layer4")

history_partial = train_and_evaluate(
    model_partial, train_loader, val_loader, epochs=5, lr=5e-4, label="C: partial unfreeze (layer4)"
)

### D. PEFT-style Adapters (conceptual demo)

Parameter-Efficient Fine-Tuning (e.g., LoRA, Adapters) freezes the pretrained backbone
entirely and injects small trainable modules between existing layers. This preserves
the original knowledge while updating only a tiny fraction of parameters. Below is a
minimal **adapter block** (a bottleneck MLP with a residual connection) inserted right
before the classification head, in the same spirit as LoRA/Adapters — trained instead
of unfreezing any backbone weights.

In [ ]:
class AdapterHead(nn.Module):
    '''A tiny bottleneck adapter (down-project -> nonlinearity -> up-project,
    with a residual connection) followed by the target classifier. Only this
    module is trained; the backbone stays fully frozen -- the PEFT idea.'''
    def __init__(self, in_features, num_classes, bottleneck=16):
        super().__init__()
        self.down = nn.Linear(in_features, bottleneck)
        self.act = nn.ReLU()
        self.up = nn.Linear(bottleneck, in_features)
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        adapted = x + self.up(self.act(self.down(x)))  # residual adapter
        return self.classifier(adapted)

model_peft = build_target_model(pretrained=True)
in_features = model_peft.fc.in_features
model_peft.fc = AdapterHead(in_features, num_classes=10, bottleneck=16)
model_peft = model_peft.to(device)

# Freeze everything except the adapter module.
for name, param in model_peft.named_parameters():
    param.requires_grad = name.startswith("fc.")

history_peft = train_and_evaluate(
    model_peft, train_loader, val_loader, epochs=5, lr=1e-3, label="D: PEFT adapter"
)

## 6. Task & Domain Relatedness — and Negative Transfer

The roadmap's third assumption is **relatedness**: source and target must share
structural similarity, or *negative transfer* can hurt target performance.

To make this concrete, we add a **no-transfer control**: the same architecture and
the same Strategy-A training recipe (frozen backbone, head-only training), but with a
**randomly-initialized** backbone instead of ImageNet weights. If relatedness/feature
transferability genuinely helps, the pretrained-and-frozen model should learn faster
and reach higher accuracy than the randomly-initialized-and-frozen model, since a
random frozen backbone provides no useful, task-general features at all.

> **On negative transfer specifically:** it typically appears when $D_s$ and $D_t$ are
> structurally unrelated (e.g., transferring weights from a text model into a vision
> model is not even architecturally meaningful) or when the source task's decision
> boundaries actively conflict with the target task's. It is a risk to check for, not
> something we can force here with compatible image models — so we discuss it
> qualitatively after the plot below rather than construct an artificial failure case.

In [ ]:
# --- Control: randomly-initialized backbone, frozen, head-only training --------------
# This isolates the effect of the PRETRAINED features themselves (relatedness benefit)
# from the training recipe, which is otherwise identical to Strategy A.
model_random = build_target_model(pretrained=False)
for name, param in model_random.named_parameters():
    param.requires_grad = name.startswith("fc")

history_random = train_and_evaluate(
    model_random, train_loader, val_loader, epochs=5, lr=1e-3, label="Control: random init, frozen"
)

In [ ]:
# --- Comparative plot across every strategy -------------------------------------------
histories = {
    "A: frozen backbone": history_frozen,
    "B: full fine-tune": history_full,
    "C: partial unfreeze": history_partial,
    "D: PEFT adapter": history_peft,
    "Control: random init (no transfer)": history_random,
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for label, h in histories.items():
    axes[0].plot(range(1, len(h["train_loss"]) + 1), h["train_loss"], marker="o", label=label)
    axes[1].plot(range(1, len(h["val_acc"]) + 1), h["val_acc"], marker="o", label=label)

axes[0].set_title("Target training loss per epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Train loss")
axes[0].legend(fontsize=8)

axes[1].set_title("Target validation accuracy per epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Val accuracy")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\nFinal validation accuracy by strategy:")
for label, h in histories.items():
    print(f"  {label:38s}: {h['val_acc'][-1]:.3f}")

**Reading the comparison:**

- The **Control (random init)** curve should sit clearly below the pretrained
  strategies — with a frozen *random* backbone, the head is learning from
  meaningless features, which is the practical signature of low relatedness /
  low feature transferability between "no prior knowledge" and the target task.
- **A (frozen)** should learn fastest per-epoch on our very small target set, since
  it trains the fewest parameters relative to the data available — matching the
  roadmap's recommendation for small target datasets.
- **B (full fine-tune)** has the most capacity to specialize to CIFAR-10, but with
  only ~600 training images it also has the most capacity to overfit; watch the
  gap between train loss and validation accuracy across epochs.
- **C (partial unfreeze)** should land between A and B — more adaptable than a fully
  frozen backbone, safer than full fine-tuning, matching its role as a controlled
  middle-ground exception.
- **D (PEFT adapter)** should behave similarly to A in spirit (backbone untouched),
  while updating a much smaller, purpose-built module than the plain linear head.

## 7. Summary — Decision Guide

| Situation | Roadmap-aligned choice | Section |
|---|---|---|
| Target dataset is very small | Freeze backbone, train head only | 5A |
| Target dataset is large / target domain differs more from source | Full fine-tuning | 5B |
| Class imbalance or domain shift, but still want some adaptation, deliberately monitored | Partial unfreezing (last block) | 5C |
| Want to preserve source knowledge, minimize trainable parameters/compute | PEFT / Adapters | 5D |
| Source and target are unrelated | Expect **negative transfer** — prefer training from scratch or finding a more related source | Section 6 |

This notebook traced the full roadmap end-to-end: a **pretrained source model**
$\to$ an explicit **target domain/task** $\to$ **feature reuse** via network
modification $\to$ a controlled comparison of **freezing/fine-tuning** strategies
$\to$ an empirical check of **task relatedness**, with negative transfer discussed
as the failure mode all of the above assumptions are designed to avoid.